# Lab 4.3 — Chains with LangChain: Sequential, Parallel and Router Chains
**Course:** AI Agents · **Unit:** UC4 — AI Agents · **Platform:** Google Colab (T4 GPU recommended)

In Labs 4.1 and 4.2 every technique was a **pattern of LLM calls**: two-stage CoT is a *sequence*, self-consistency is *n calls in parallel plus a vote*, and a forum assistant that handles math, code and concept questions differently needs a *router*. In this lab we build those patterns explicitly with the **LangChain Expression Language (LCEL)**, so they become components with one interface (`invoke`, `batch`, `stream`) that an agent can compose, debug and make robust.

### Learning objectives
1. Compose **sequential chains** that pass outputs forward, keep intermediate results (`RunnablePassthrough.assign`), and can be inspected and debugged.
2. Make a chain step **robust** with `with_retry` and `with_fallbacks`, and know when each one helps.
3. Build **parallel chains** with `RunnableParallel` (fan-out over *tasks*) and `.map()` (fan-out over *data*), merge their outputs, and measure the speed-up.
4. Build a **router chain** that classifies each input and dispatches it to the right specialised chain, and evaluate the router **separately** from the answers.

### Lab structure
| Part | Topic | What you do |
|---|---|---|
| 0 | Setup and the building blocks from Labs 3.3–3.4 | Run |
| 1 | Sequential chains, inspection and robustness | Run, then **Exercise 1** |
| 2 | Parallel chains and map-reduce | Run, then **Exercise 2** |
| 3 | Router chains | Run, then **Exercise 3** |
| 4 | Questions | Answer |

### Running this notebook in Google Colab
1. **File → Upload notebook**, then **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells **in order**. Part 0 installs Ollama and downloads the model (≈ 2 minutes).

> **On your own laptop:** install Ollama from <https://ollama.com/download>, skip cell 0.1 and run the rest unchanged.

## Part 0 — Setup
Same setup as Labs 4.1 and 4.2. `OLLAMA_NUM_PARALLEL=4` matters more than ever in this lab: it lets the server answer the branches of a parallel chain **at the same time**.

In [2]:
#%% 0.1 Install Ollama and start the server (once per Colab session, ~1 minute)
import os, subprocess, time, urllib.request

!sudo apt-get -qq update > /dev/null 2>&1 && sudo apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"      # answer up to 4 requests at once (parallel branches, .batch())
ollama_log = open("ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)

def wait_for_ollama(timeout=60):
    # Poll the local API until the server answers, or give up after `timeout` seconds.
    for _ in range(timeout):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
            return True
        except Exception:
            time.sleep(1)
    return False

print("✅ Ollama server is running" if wait_for_ollama() else "❌ Server did not start:\n" + open("ollama.log").read()[-2000:])

✅ Ollama server is running


In [3]:
#%% 0.2 Download the model and install LangChain
MODEL = "llama3.2:3b"        # faster alternative: "qwen2.5:1.5b"
!ollama pull {MODEL} > /dev/null 2>&1
!ollama list
%pip install -q -U langchain-core langchain-ollama

NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


In [4]:
#%% 0.3 Imports and models
import re, textwrap, time
from collections import Counter
from operator import itemgetter

import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL, temperature=0, num_predict=512, num_ctx=4096)
llm_sample = ChatOllama(model=MODEL, temperature=0.7, top_p=0.95, num_predict=512, num_ctx=4096)
print(llm.invoke("Say 'ready' and nothing else.").content)

ready


### 0.4 Building blocks from Labs 4.1–4.2
We reuse the CoT prompt with its **answer contract** and the answer extractor from Lab 4.2, and a few of its math problems.

In [5]:
#%% 0.4 Reused pieces
COT_SYSTEM = ("You solve math word problems. Think step by step and write each calculation on its own line. "
              "Finish with a last line of the form 'Answer: <number>'.")
cot_prompt = ChatPromptTemplate.from_messages([("system", COT_SYSTEM), ("human", "{question}")])

def extract_answer(text):
    # Number after the last 'Answer:' (or the last number in the text); None if there is no number.
    hits = re.findall(r"answer[^0-9\-\n]*(-?\d[\d,]*(?:\.\d+)?)", text, flags=re.IGNORECASE)
    if not hits:
        hits = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
    if not hits:
        return None
    try:
        value = float(hits[-1].replace(",", ""))
        return int(value) if value.is_integer() else value
    except ValueError:
        return None

PROBLEMS = [
    ("A school has 3 buses with 42 seats each, all full, and a fourth bus with 17 students. How many students are travelling?", 143),
    ("Pedro is 3 times as old as his daughter. In 12 years he will be twice as old as her. How old is Pedro now?", 36),
    ("A phone costs $480. It is discounted 15%, then a 12% tax is added to the discounted price. What is the final price, rounded to the nearest dollar?", 457),
]

def show(d, width=110):
    # Pretty-print a dict of chain outputs, one key per block.
    for k, v in d.items():
        print(f"━━━ {k} ━━━\n" + textwrap.fill(str(v), width, replace_whitespace=False) + "\n")

---
## Part 1 — Sequential chains

### 1.1 What `|` really builds
`a | b` creates a **`RunnableSequence`**: the output of `a` becomes the input of `b`. Two shortcuts make sequences easy to write — LangChain **coerces** them into Runnables automatically:

| You write | LangChain builds | Meaning |
|---|---|---|
| `{"x": chain1, "y": itemgetter("y")}` | `RunnableParallel` | run every value on the same input, return a dict |
| `lambda d: ...` or a function | `RunnableLambda` | any Python step (parsing, formatting, a tool call) |

The most common sequence: step 1 produces text, a **dict literal** gives that text a name, and the step-2 prompt uses the name as a variable.

In [6]:
#%% 1.1 A simple sequence: topic → learning objectives → quiz questions
objectives_prompt = ChatPromptTemplate.from_messages([
    ("system", "You design university courses. Be concise."),
    ("human", "Write 3 learning objectives for a 1-hour class on {topic}. One line each, starting with a verb."),
])
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write multiple-choice quiz questions."),
    ("human", "Write one quiz question with 4 options (mark the correct one) for EACH of these objectives:\n{objectives}"),
])

objectives_chain = objectives_prompt | llm | StrOutputParser()
quiz_chain = {"objectives": objectives_chain} | quiz_prompt | llm | StrOutputParser()   # dict → RunnableParallel

print(quiz_chain.invoke({"topic": "recursion in Python"}))

Here are the quiz questions for each objective:

**Objective 1: Analyze the basic structure of recursive functions in Python.**

What is the typical structure of a recursive function in Python?

A) A single loop that continues until a condition is met
B) A function definition with a base case and a recursive call
C) A class definition with an instance variable and a method
D) A conditional statement with an if-else block

**Correct answer:** B) A function definition with a base case and a recursive call

**Objective 2: Identify and explain the key differences between recursive and iterative approaches.**

What is a key difference between recursive and iterative approaches to solving a problem?

A) Recursion uses more memory than iteration
B) Recursion is faster than iteration
C) Recursion uses a loop to repeat the process, while iteration uses a recursive call
D) Recursion is only used for problems that can be solved with loops

**Correct answer:** C) Recursion uses a recursive call to

The sequence works, but it **throws away** the intermediate result: we only see the quiz, not the objectives it was built from. In an agent you usually want every intermediate value — to show it, to log it, or to debug a wrong final answer.

### 1.2 Keep everything: `RunnablePassthrough.assign`
`RunnablePassthrough.assign(key=chain)` runs `chain` on the current dict and **adds** its output under `key`, keeping all existing keys. A chain of `.assign` steps works like a growing record:

```
{"topic"} ──assign──► {"topic", "objectives"} ──assign──► {"topic", "objectives", "quiz"}
```

In [7]:
#%% 1.2 A sequence that keeps its intermediate results
lesson_chain = (
    RunnablePassthrough.assign(objectives=objectives_chain)
    | RunnablePassthrough.assign(quiz=quiz_prompt | llm | StrOutputParser())
)
lesson = lesson_chain.invoke({"topic": "list comprehensions in Python"})
print("keys:", list(lesson.keys()), "\n")
show(lesson)

keys: ['topic', 'objectives', 'quiz'] 

━━━ topic ━━━
list comprehensions in Python

━━━ objectives ━━━
Here are three learning objectives for a 1-hour class on list comprehensions in Python:

1. Analyze the syntax
and structure of list comprehensions in Python.
2. Write efficient and readable list comprehensions to
transform and filter data.
3. Evaluate the performance and trade-offs of using list comprehensions versus
traditional for loops.

━━━ quiz ━━━
Here are the quiz questions for each objective:

**Objective 1: Analyze the syntax and structure of list
comprehensions in Python.**

What is the correct syntax for a list comprehension in Python?

A) `my_list =
[expression for variable in iterable if condition]`
B) `my_list = variable for variable in iterable if
condition`
C) `my_list = expression for variable in iterable if condition`
D) `my_list = [expression for
variable in iterable]`

**Correct answer:** A) `my_list = [expression for variable in iterable if condition]`
**Objecti

### 1.3 Inspecting a chain
Two tools help when a chain grows:
- `chain.get_graph().draw_mermaid()` prints the structure as a Mermaid diagram (paste it into <https://mermaid.live> to see it);
- `ConsoleCallbackHandler` prints **every step's input and output** while the chain runs — the first thing to try when the final answer is wrong.

In [8]:
#%% 1.3 Structure and step-by-step trace
from langchain_core.tracers.stdout import ConsoleCallbackHandler

print(lesson_chain.get_graph().draw_mermaid())

small = {"objectives": objectives_chain} | RunnableLambda(lambda d: d["objectives"].count("\n") + 1)
print("number of objective lines:", small.invoke({"topic": "dictionaries"},
                                                  config={"callbacks": [ConsoleCallbackHandler()]}))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	Parallel\3cobjectives\3eInput([Parallel<objectives>Input]):::first
	Parallel\3cobjectives\3eOutput(Parallel<objectives>Output)
	ChatPromptTemplate_1(ChatPromptTemplate)
	ChatOllama_1(ChatOllama)
	StrOutputParser_1(StrOutputParser)
	Passthrough_1(Passthrough)
	Parallel\3cquiz\3eInput(Parallel<quiz>Input)
	Parallel\3cquiz\3eOutput([Parallel<quiz>Output]):::last
	ChatPromptTemplate_2(ChatPromptTemplate)
	ChatOllama_2(ChatOllama)
	StrOutputParser_2(StrOutputParser)
	Passthrough_2(Passthrough)
	ChatPromptTemplate_1 --> ChatOllama_1;
	ChatOllama_1 --> StrOutputParser_1;
	Parallel\3cobjectives\3eInput --> ChatPromptTemplate_1;
	StrOutputParser_1 --> Parallel\3cobjectives\3eOutput;
	Parallel\3cobjectives\3eInput --> Passthrough_1;
	Passthrough_1 --> Parallel\3cobjectives\3eOutput;
	ChatPromptTemplate_2 --> ChatOllama_2;
	ChatOllama_2 --> StrOutputParser_2;
	Parallel\3cquiz\3eInput --> ChatPromptTemplate_2;
	StrOutputParser_2 --> Paralle

### 1.4 Lab 3.4 patterns as sequences
Two-stage CoT (Lab 4.2, Part 1.3) is a two-step sequence. With `.assign` we keep the reasoning **and** the extracted number in one result, and a final `RunnableLambda` turns the extraction into a value.

In [9]:
#%% 1.4 Two-stage CoT as a sequence that keeps every step
reason_prompt = ChatPromptTemplate.from_messages([("human", "{question}\n\nLet's think step by step.")])
extract_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}\n\nReasoning:\n{reasoning}\n\nTherefore, the final answer (a number only) is:")])

two_stage = (
    RunnablePassthrough.assign(reasoning=reason_prompt | llm | StrOutputParser())
    | RunnablePassthrough.assign(raw=extract_prompt | llm | StrOutputParser())
    | RunnablePassthrough.assign(answer=lambda d: extract_answer(d["raw"]))
)
for q, gold in PROBLEMS:
    r = two_stage.invoke({"question": q})
    print(f"answer = {r['answer']} | gold = {gold} | reasoning: {textwrap.shorten(r['reasoning'], 120)}")

answer = 143 | gold = 143 | reasoning: To find the total number of students traveling, we need to calculate the number of students on the three full [...]
answer = 36 | gold = 36 | reasoning: Let's break it down step by step. Let Pedro's current age be P and his daughter's current age be D. Since Pedro is [...]
answer = 457 | gold = 457 | reasoning: To find the final price, we'll calculate the discount and tax step by step. 1. Calculate the discount: Discount = [...]


### 1.5 Robust steps: `with_retry` and `with_fallbacks`
In a sequence, **one failing step breaks everything after it**. Two wrappers make a step robust:

| Wrapper | What it does | Use it when |
|---|---|---|
| `step.with_retry(stop_after_attempt=3)` | runs the **same** step again if it raises an exception | the failure is random: a timeout, a sampled output that broke the format |
| `step.with_fallbacks([other])` | runs a **different** step if the first one raises | the step can fail systematically: another prompt, model or method |

Both react to **exceptions**, so a step must *raise* when its output is unusable. Below, `strict_number` raises when the `Answer:` line is missing.

> ⚠️ Retrying a `temperature=0` call usually returns the **same** broken output. A retry helps only if something changes between attempts — for example sampling with `llm_sample`.

In [10]:
#%% 1.5 A strict parser, retries and a fallback
def strict_number(msg):
    # Accept only an explicit 'Answer: <number>' line; raise otherwise, so retries/fallbacks can react.
    hits = re.findall(r"^\s*\**answer\**\s*:\s*\**\s*\$?\s*(-?\d[\d,]*(?:\.\d+)?)", msg.content, flags=re.I | re.M)
    if not hits:
        raise ValueError("contract broken: no 'Answer:' line")
    value = float(hits[-1].replace(",", ""))
    return int(value) if value.is_integer() else value

# A deliberately bad prompt: it asks for prose, so the contract is usually broken
bad_prompt = ChatPromptTemplate.from_messages([("human", "{question}\nAnswer in one friendly sentence.")])

strict_solver = bad_prompt | llm | RunnableLambda(strict_number)
try:
    print("strict:", strict_solver.invoke({"question": PROBLEMS[1][0]}))
except ValueError as e:
    print("strict solver failed →", e)

fallback = two_stage | RunnableLambda(lambda d: d["answer"])
robust_solver = (bad_prompt | llm_sample | RunnableLambda(strict_number)) \
    .with_retry(stop_after_attempt=2) \
    .with_fallbacks([fallback])
print("robust:", robust_solver.invoke({"question": PROBLEMS[1][0]}), "| gold:", PROBLEMS[1][1])

strict solver failed → contract broken: no 'Answer:' line
robust: 36 | gold: 36


### ✏️ Exercise 1 — A feedback pipeline for student code
Build `feedback_chain`, a **three-step sequence** that keeps every intermediate result:
1. `explanation` — explain in 2–3 sentences what the code is meant to do;
2. `issues` — using the code **and** the explanation, list at most 3 bugs or problems (numbered);
3. `message` — write a short, encouraging feedback message to the student **in `{language}`** using the issues; fix the language to Spanish with `.partial()`.

Run it on the two `STUDENT_CODE` snippets with **one** `.batch()` call and print all keys. Then answer: if step 1 misunderstands the code, what happens to steps 2 and 3? How would you notice?

In [11]:
#%% Exercise 1 — inputs (run this cell)
STUDENT_CODE = [
    "def is_even(n):\n    if n % 2 == 1:\n        return True\n    return False",
    "def total(prices):\n    s = 0\n    for i in range(1, len(prices)):\n        s += prices[i]\n    return s",
]

In [12]:
#%% Exercise 1 — your solution
# explain_prompt = ChatPromptTemplate.from_messages([...])          # uses {code}
# issues_prompt = ChatPromptTemplate.from_messages([...])           # uses {code} and {explanation}
# message_prompt = ChatPromptTemplate.from_messages([...]).partial(language="Spanish")   # uses {issues}, {language}
#
# feedback_chain = (
#     RunnablePassthrough.assign(explanation=...)
#     | RunnablePassthrough.assign(issues=...)
#     | RunnablePassthrough.assign(message=...)
# )
# results = feedback_chain.batch([{"code": c} for c in STUDENT_CODE])
# for r in results:
#     show(r)

---
## Part 2 — Parallel chains

### 2.1 Fan-out over tasks: `RunnableParallel`
When several steps need the **same input** and do **not depend on each other**, run them at the same time. `RunnableParallel({...})` sends the input to every branch concurrently and returns a dict with one key per branch. (A dict literal inside a chain is the same thing.)

Example: an assistant that reads a forum post and needs, at once, a summary, the category, the urgency and the questions it asks.

In [13]:
#%% 2.1 Four analyses of one post, in parallel
POST = ("Hi everyone, I've been stuck for two days on lab 3.4. When I run the self-consistency cell, Colab "
        "disconnects after a few minutes and I lose everything. I tried the smaller model but my accuracy "
        "got much worse. The deadline is on Sunday and I still have exercises 2 and 3 to do. Is it OK to "
        "run only 3 samples instead of 5? And does the instructor accept a late submission in this case?")

def text_chain(instruction):
    # A one-step chain: instruction + the post → string.
    p = ChatPromptTemplate.from_messages([("system", "Answer briefly."), ("human", instruction + "\n\nPost:\n{post}")])
    return p | llm | StrOutputParser()

branches = {
    "summary":   text_chain("Summarize the post in one sentence."),
    "category":  text_chain("Classify the post as one of: deadline, grading, concept_doubt, technical_issue, other. "
                            "Answer with the category only."),
    "urgency":   text_chain("Rate how urgent the post is: low, medium or high. Answer with one word and a short reason."),
    "questions": text_chain("List the explicit questions the student asks, one per line."),
}
analysis = RunnableParallel(branches)

t0 = time.time(); seq = {k: c.invoke({"post": POST}) for k, c in branches.items()}; t_seq = time.time() - t0
t0 = time.time(); par = analysis.invoke({"post": POST}); t_par = time.time() - t0
print(f"one by one: {t_seq:.1f} s | RunnableParallel: {t_par:.1f} s\n")
show(par)

one by one: 2.0 s | RunnableParallel: 1.5 s

━━━ summary ━━━
The student is struggling with Lab 3.4 in Colab, experiencing frequent disconnections and accuracy issues, and
is uncertain about whether to submit with fewer samples and whether a late submission is acceptable.

━━━ category ━━━
other

━━━ urgency ━━━
High. Deadline is on Sunday and the student is struggling with the assignment.

━━━ questions ━━━
Here are the explicit questions the student asks:

1. Is it OK to run only 3 samples instead of 5?
2. Does the
instructor accept a late submission in this case?



### 2.2 Merge the branches
A parallel step is usually followed by a **merge** step that combines the branch outputs: a Python function (a vote, a dict, a database row) or another prompt. Here a prompt writes a reply draft for the teaching assistant from the four analyses — note that the merge prompt also gets the original post through `itemgetter`.

In [14]:
#%% 2.2 Parallel analyses + a merge prompt
merge_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a teaching assistant. Write a short, kind reply (at most 90 words) to a student's forum post."),
    ("human", "Post:\n{post}\n\nSummary: {summary}\nCategory: {category}\nUrgency: {urgency}\n"
              "Questions to answer:\n{questions}\n\nCourse policy: late submissions lose 10% per day; "
              "using 3 samples instead of 5 is acceptable if it is stated in the notebook."),
])
reply_chain = RunnablePassthrough.assign(**branches) | merge_prompt | llm | StrOutputParser()
print(reply_chain.invoke({"post": POST}))

Hi there, I can see you're really struggling with Lab 3.4. Don't worry, it's normal to encounter issues with Colab disconnections. Regarding your questions, yes, it's perfectly fine to run only 3 samples instead of 5, as long as you mention it in your notebook. However, I would recommend trying to complete the exercises before submitting, as it's always better to get the full 5 samples. If you're still having trouble, you can discuss with your instructor about a possible late submission. Good luck with the rest of the assignment!


`RunnablePassthrough.assign(**branches)` runs all branches **in parallel** and adds their outputs to the input dict — a compact way to write "fan out, keep the input, then merge".

### 2.3 Self-consistency as a parallel chain
In Lab 4.2 we wrote self-consistency as a Python function. As a chain it is a `RunnableParallel` of `n` copies of the sampled CoT chain followed by a vote. The result is a Runnable like any other: it can be piped, batched, routed to — and used as a step of a bigger chain.

In [15]:
#%% 2.3 Self-consistency = parallel sampling + a vote
def make_self_consistency(chain, n=5):
    # n parallel samples of `chain`, then a majority vote with its agreement.
    paths = RunnableParallel({f"path_{i}": chain for i in range(n)})

    def vote(outputs):
        answers = [extract_answer(o.content) for o in outputs.values()]
        votes = Counter(a for a in answers if a is not None)
        if not votes:
            return {"answer": None, "agreement": 0.0, "answers": answers}
        answer, count = votes.most_common(1)[0]
        return {"answer": answer, "agreement": count / n, "answers": answers}

    return paths | RunnableLambda(vote)

sc_solver = make_self_consistency(cot_prompt | llm_sample, n=5)
for q, gold in PROBLEMS:
    r = sc_solver.invoke({"question": q})
    print(f"answers = {r['answers']} → {r['answer']} (agreement {r['agreement']:.0%}) | gold = {gold}")

answers = [143, 143, 143, 143, 143] → 143 (agreement 100%) | gold = 143
answers = [36, 36, 36, 36, 36] → 36 (agreement 100%) | gold = 36
answers = [457, 457, 457, 457, 457] → 457 (agreement 100%) | gold = 457


### 2.4 Fan-out over data: map-reduce
`RunnableParallel` runs **different tasks** on one input. The other kind of parallelism runs **the same task** on many pieces of data: `chain.map()` turns a chain that takes one input into a chain that takes a **list** (it runs them concurrently, like `.batch()`).

**Map-reduce summarization** is the classic use: a document too long for the context window is split into chunks, each chunk is summarized in parallel (*map*), and the partial summaries are combined (*reduce*).

In [16]:
#%% 2.4 Map-reduce summarization of a long text
LONG_TEXT = '''An intelligent agent perceives its environment and acts on it to reach a goal. Classical agents were written by hand: a programmer listed the rules, the states and the actions. Reinforcement learning changed this by letting an agent learn a policy from rewards, as we saw with REINFORCE.

Large language models added a new kind of agent. Instead of a policy network trained for one task, the agent uses a general model that reads instructions written in natural language. The prompt becomes the main way to program the agent, which is why prompt templates and few-shot examples matter.

Reasoning prompts make these agents more reliable. Chain-of-thought gives the model more computation per answer, self-consistency samples several reasoning paths and votes, and self-refine lets the model improve a draft with feedback. Each of these techniques is a pattern of several model calls.

Chains turn these patterns into software components. A sequential chain passes outputs forward, a parallel chain runs independent steps at the same time, and a router chooses which specialised chain should handle an input. Each chain has the same interface, so chains can be nested.

The next step is to give the agent tools and memory. Tools let the model act outside the text, for example by running code or searching documents, and they also provide trustworthy feedback. Memory lets the agent keep track of a conversation. Together they turn a chain into an agent that decides its own next step.'''

chunks = [p for p in LONG_TEXT.split("\n\n") if p.strip()]
summarize_one = (ChatPromptTemplate.from_messages([("human", "Summarize this paragraph in one sentence:\n{chunk}")])
                 | llm | StrOutputParser())
reduce_prompt = ChatPromptTemplate.from_messages([
    ("human", "Combine these partial summaries into one coherent paragraph of at most 80 words:\n{summaries}")])

map_reduce = (
    RunnableLambda(lambda text: [{"chunk": c} for c in text.split("\n\n") if c.strip()])   # split
    | summarize_one.map()                                                                   # map (parallel)
    | RunnableLambda(lambda parts: {"summaries": "\n".join(f"- {p}" for p in parts)})       # join
    | reduce_prompt | llm | StrOutputParser()                                              # reduce
)
print(f"{len(chunks)} chunks\n")
print(map_reduce.invoke(LONG_TEXT))

5 chunks

Reinforcement learning and large language models have transformed intelligent agent design, enabling agents to learn policies through rewards and general models to perform tasks. Reasoning prompts, such as chain-of-thought and self-refine, enhance reliability by exploring multiple paths and improving responses through feedback. Chains process patterns by passing outputs forward, running steps in parallel, or routing inputs, allowing for nesting and combination. Tools and memory enable self-sustaining decision-making processes.


👉 **Think about it:** each chunk is summarized **without seeing the others**. What could go wrong if paragraph 4 refers to something defined in paragraph 2?

### ✏️ Exercise 2 — Parallel reviewers and a meta-review
Build `review_chain` for a short research abstract (`ABSTRACT` below):
1. Three **reviewer** branches that run in parallel, each with its own system prompt: *methodology* (is the evaluation convincing?), *clarity* (is it understandable for a non-expert?), *novelty* (what is new?). Each returns at most 3 bullet points and a score from 1 to 5 on a last line `Score: <n>`.
2. A **merge** step: a Python function that extracts the three scores and computes their mean, and a **meta-reviewer** prompt that writes a 60-word decision (accept / revise / reject) from the three reviews.
3. Return a dict with the three reviews, the scores, the mean and the decision.
4. Measure the time of the parallel version against calling the three reviewers one by one.

In [17]:
#%% Exercise 2 — input (run this cell)
ABSTRACT = ("We propose a prompt-based agent that answers students' questions in a university programming course. "
            "The agent routes each question to a specialised prompt (code review, concept explanation or math) "
            "and uses self-consistency to decide when to ask a teaching assistant instead of answering. "
            "In a pilot with 40 students over 4 weeks, the agent answered 72% of the questions without human help, "
            "and students rated its answers 4.1 out of 5. We discuss limitations, including the small sample and "
            "the lack of a control group.")

In [18]:
#%% Exercise 2 — your solution
# def reviewer(focus):
#     p = ChatPromptTemplate.from_messages([("system", ...), ("human", "Abstract:\n{abstract}")])
#     return p | llm | StrOutputParser()
#
# reviewers = {"methodology": reviewer(...), "clarity": reviewer(...), "novelty": reviewer(...)}
#
# def add_scores(d):
#     ...  # extract the three 'Score: n' values and their mean
#
# meta_prompt = ChatPromptTemplate.from_messages([...])
# review_chain = (RunnablePassthrough.assign(**reviewers)
#                 | RunnableLambda(add_scores)
#                 | RunnablePassthrough.assign(decision=meta_prompt | llm | StrOutputParser()))

---
## Part 3 — Router chains

A course assistant receives very different questions. One prompt for everything is a compromise; a better design **classifies** each question and **dispatches** it to a specialised chain:

```
                     ┌─► math    → CoT with answer contract (Lab 4.2)
question ─► classify ┼─► code    → code reviewer            (Lab 4.1)
                     ├─► concept → tutor explanation        (Lab 4.1)
                     └─► other   → general assistant (default)
```

A router has two parts that can fail **independently**: the **classifier** (did the question go to the right place?) and the **routes** (was the answer good?). We evaluate them separately.

In [19]:
#%% 3.1 The classifier: few-shot + label parsing (Lab 3.3)
ROUTE_LABELS = ["math", "code", "concept", "other"]
ROUTER_SYSTEM = ("You route questions from students of an AI and programming course. Categories:\n"
                 "math: a word problem or calculation with a numeric answer\n"
                 "code: a question about a specific piece of code (bugs, errors, improvements)\n"
                 "concept: a question about how an idea or technique works\n"
                 "other: anything else\n"
                 "Answer with the category name only.")
ROUTER_EXAMPLES = [
    {"input": "A train travels 240 km in 3 hours. What is its average speed in km/h?", "output": "math"},
    {"input": "Why does my loop never stop?\nwhile i < 10:\n    print(i)", "output": "code"},
    {"input": "What is the difference between a list and a tuple?", "output": "concept"},
    {"input": "Where is the classroom for tomorrow?", "output": "other"},
    {"input": "If 3 pens cost $4.50, how much do 7 pens cost?", "output": "math"},
    {"input": "What does the discount factor do in reinforcement learning?", "output": "concept"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
router_prompt = ChatPromptTemplate.from_messages([
    ("system", ROUTER_SYSTEM),
    FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=ROUTER_EXAMPLES),
    ("human", "{question}"),
])

def parse_route(text):
    # First route label mentioned in the output; 'other' if none (a safe default).
    t = text.strip().lower()
    hits = [(t.find(label), label) for label in ROUTE_LABELS if label in t]
    return min(hits)[1] if hits else "other"

classifier = router_prompt | llm | StrOutputParser() | RunnableLambda(parse_route)
print(classifier.invoke({"question": "How many legs do 7 cows and 5 chickens have?"}))

math


In [20]:
#%% 3.2 The specialised routes
math_route = cot_prompt | llm | StrOutputParser()

code_route = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly code reviewer for first-year students. Explain the problem, then show the fixed "
               "code. Be brief."),
    ("human", "{question}"),
]) | llm | StrOutputParser()

concept_route = ChatPromptTemplate.from_messages([
    ("system", "You are a patient tutor. Explain the concept in at most 4 sentences and give one concrete example."),
    ("human", "{question}"),
]) | llm | StrOutputParser()

other_route = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant for a university course. Answer briefly. If the question needs "
               "course-specific information you do not have, say so and suggest asking the teaching assistant."),
    ("human", "{question}"),
]) | llm | StrOutputParser()

ROUTES = {"math": math_route, "code": code_route, "concept": concept_route, "other": other_route}

### 3.3 The router: a function that returns a chain
The recommended way to route in LCEL is a `RunnableLambda` whose function **returns a Runnable**: LangChain then runs that Runnable on the same input. The classifier's output is added first with `.assign`, so the final result shows **where** each question went and **what** it answered.

In [21]:
#%% 3.3 Router = classify, then dispatch
def choose_route(d):
    return ROUTES.get(d["route"], other_route)       # unknown label → default route

router = (RunnablePassthrough.assign(route=classifier)
          | RunnablePassthrough.assign(answer=RunnableLambda(choose_route)))

for q in ["How many legs do 7 cows and 5 chickens have?",
          "What is an embedding?",
          "Why does this print None?\ndef square(x):\n    y = x * x\nprint(square(3))"]:
    r = router.invoke({"question": q})
    print(f"━━━ route = {r['route']} ━━━\nQ: {q}\nA: {textwrap.shorten(r['answer'], 300)}\n")

━━━ route = math ━━━
Q: How many legs do 7 cows and 5 chickens have?
A: To find the total number of legs, we need to calculate the legs of each animal and then add them together. Cows have 4 legs each, so 7 cows have 7 x 4 = 28 legs. Chickens have 2 legs each, so 5 chickens have 5 x 2 = 10 legs. Now, let's add the total number of legs together: 28 + 10 = 38 legs. [...]

━━━ route = concept ━━━
Q: What is an embedding?
A: An embedding is a way to represent a high-dimensional space in a lower-dimensional space, while trying to preserve the relationships and distances between the points in the original space. This is often used in machine learning and data analysis to simplify complex data. For example, imagine [...]

━━━ route = code ━━━
Q: Why does this print None?
def square(x):
    y = x * x
print(square(3))
A: **Problem:** The issue lies in the fact that the `print` statement is not inside the `square` function. In Python, the `print` function is a built-in function that returns `Non

### 3.4 The same router with `RunnableBranch`
`RunnableBranch((condition, runnable), ..., default)` checks the conditions **in order** and runs the first runnable whose condition is true. It is handy when the conditions are rules (keywords, lengths, flags) rather than a label lookup.

In [22]:
#%% 3.4 RunnableBranch: rules first, then the label
branch = RunnableBranch(
    (lambda d: "def " in d["question"] or "\n    " in d["question"], code_route),   # a rule: looks like code
    (lambda d: d["route"] == "math", math_route),
    (lambda d: d["route"] == "concept", concept_route),
    other_route,                                                                   # default
)
router_b = RunnablePassthrough.assign(route=classifier) | RunnablePassthrough.assign(answer=branch)
r = router_b.invoke({"question": "Is there a bug here?\nfor i in range(1, len(xs)):\n    total += xs[i]"})
print("route label:", r["route"], "\n" + textwrap.shorten(r["answer"], 300))

route label: code 
**Bug: Index Out of Bounds** The bug in this code is that the loop starts from index 1, but it should start from index 0. In Python, list indices start at 0, not 1. **Fixed Code:** ```python for i in range(0, len(xs)): total += xs[i] ``` Alternatively, you can use the `enumerate` function to [...]


### 3.5 Evaluate the router on its own
Routing errors are invisible in the final answer — a math question sent to the general route still gets *an* answer, just a worse one. So we measure **routing accuracy** on labelled questions, and look at the **confusion table** to see which routes get mixed up.

In [23]:
#%% 3.5 Routing accuracy and confusion table
ROUTER_TEST = [
    ("A tank holds 500 liters and is 40% full. A pump adds 15 liters per minute. How many minutes until it is full?", "math"),
    ("Luis reads 18 pages on day one and 4 more each day. How many pages after 5 days?", "math"),
    ("A phone costs $480, discounted 15%, then 12% tax. Final price?", "math"),
    ("Why does this function return None?\ndef add(a, b):\n    c = a + b", "code"),
    ("Is there a bug here?\nfor i in range(1, len(xs)):\n    total += xs[i]", "code"),
    ("How can I make this faster?\nresult = []\nfor x in xs:\n    result.append(x * x)", "code"),
    ("What is the difference between supervised and reinforcement learning?", "concept"),
    ("Why does a higher temperature make LLM outputs more random?", "concept"),
    ("What is an embedding?", "concept"),
    ("Can you recommend a good podcast about AI?", "other"),
    ("What time does the lab session start on Friday?", "other"),
    ("Thanks for the help yesterday!", "other"),
]
preds = classifier.batch([{"question": q} for q, _ in ROUTER_TEST], config={"max_concurrency": 4})
route_df = pd.DataFrame({"question": [textwrap.shorten(q, 60) for q, _ in ROUTER_TEST],
                         "gold": [g for _, g in ROUTER_TEST], "pred": preds})
route_df["correct"] = route_df.gold == route_df.pred
print(f"routing accuracy = {route_df.correct.mean():.0%}")
display(route_df)
display(pd.crosstab(route_df.gold, route_df.pred))

routing accuracy = 100%


,question,gold,pred,correct
0,A tank holds 500 liters and is 40% full. A pum...,math,math,True
1,Luis reads 18 pages on day one and 4 more each...,math,math,True
2,"A phone costs $480, discounted 15%, then 12% t...",math,math,True
3,"Why does this function return None? def add(a,...",code,code,True
4,"Is there a bug here? for i in range(1, len(xs)...",code,code,True
5,How can I make this faster? result = [] for x ...,code,code,True
6,What is the difference between supervised and ...,concept,concept,True
7,Why does a higher temperature make LLM outputs...,concept,concept,True
8,What is an embedding?,concept,concept,True
9,Can you recommend a good podcast about AI?,other,other,True


pred,code,concept,math,other
gold,,,,
code,3,0,0,0
concept,0,3,0,0
math,0,0,3,0
other,0,0,0,3


👉 Which route receives the mistakes? A good default route (`other_route` says *"I don't know, ask the TA"*) makes a routing error **cheaper**. A bad default (for example the math route) would make it expensive.

### ✏️ Exercise 3 — Add a route with course information
The question *"What time does the lab session start on Friday?"* cannot be answered by any route: it needs **course information**.
1. Add a new label `admin` (schedules, deadlines, grading policy) to the classifier: update `ROUTE_LABELS`, `ROUTER_SYSTEM` and add **two** new examples (not copies of test questions).
2. Write `admin_route`: a prompt that answers **only** from `COURSE_INFO` (below) and says *"I don't know, please ask the teaching assistant"* otherwise (the idea of `rlm/rag-prompt` from Lab 3.3).
3. Add it to `ROUTES`, rebuild the classifier and the router, relabel the test set (the Friday question and one new admin question of your own become `admin`), and report routing accuracy **before and after**.
4. Make the new route robust: wrap `admin_route` with `.with_fallbacks([other_route])`. When would that fallback actually run?

In [24]:
#%% Exercise 3 — course information (run this cell)
COURSE_INFO = ("AI Agents course. Lab sessions: Fridays from 10:00 to 12:00 in room B-204. "
               "Lab submissions are due on Sundays at 23:59 (Ecuador time). Late submissions lose 10% per day "
               "and are not accepted after 3 days. Labs are 40% of the final grade, the midterm 25% and the project 35%.")

In [25]:
#%% Exercise 3 — your solution
# ROUTE_LABELS = [...]
# ROUTER_SYSTEM = (...)
# ROUTER_EXAMPLES = ROUTER_EXAMPLES + [...]
# router_prompt = ChatPromptTemplate.from_messages([...])
# classifier = router_prompt | llm | StrOutputParser() | RunnableLambda(parse_route)
#
# admin_route = ChatPromptTemplate.from_messages([...]).partial(info=COURSE_INFO) | llm | StrOutputParser()
# ROUTES["admin"] = admin_route.with_fallbacks([other_route])
#
# ROUTER_TEST_2 = [...]     # relabelled test set
# ... evaluate as in 3.5

---
## Part 4 — Questions
Answer briefly (2–4 sentences each), using your outputs as evidence.

**Q1.** What does LangChain build when you put a **dict literal** or a **lambda** inside a chain? Why did we prefer `RunnablePassthrough.assign` to a plain sequence in Parts 1.2 and 1.4?

**Q2.** In Exercise 1, what happens to the final message when step 1 is wrong? Name two techniques from this lab that help you find the failing step.

**Q3.** Explain the difference between `with_retry` and `with_fallbacks`. Why is retrying a `temperature=0` step usually useless, and why must a step **raise** for either wrapper to help?

**Q4.** Report the times you measured in 2.1 and Exercise 2. Why is the speed-up smaller than the number of branches? Give one situation where `RunnableParallel` is the right choice and one where `.map()` is.

**Q5.** Why do we evaluate the router separately from the answers? Using your confusion table, what is the cost of a routing error, and how does the choice of the **default route** change it?

**Q6.** Which pattern from Lab 3.4 could **not** be written as a pure LCEL chain in this lab, and why? What is still missing for our course assistant to remember a conversation?

**Your answers:**

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**
- **Q5.**
- **Q6.**

---
### What's next
**Lab D** adds **memory**: `RunnableWithMessageHistory` fills the `MessagesPlaceholder` from Lab 3.3 automatically, session by session. The capstone then combines everything into a conversational course assistant: a router (Part 3), CoT with self-consistency for math (Lab 3.4), self-refine for written explanations, and memory.

### References
- LangChain documentation — LCEL and runnables (`RunnableSequence`, `RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`, `RunnableBranch`), routing, retries and fallbacks: <https://docs.langchain.com>
- T. Kojima et al. *Large Language Models are Zero-Shot Reasoners.* NeurIPS 2022 (two-stage CoT).
- X. Wang et al. *Self-Consistency Improves Chain of Thought Reasoning in Language Models.* ICLR 2023.
- J. Dean, S. Ghemawat. *MapReduce: Simplified Data Processing on Large Clusters.* OSDI 2004 (the map-reduce pattern).
- Ollama: <https://ollama.com> · Mermaid live editor: <https://mermaid.live>